## TP 10 Détection des mentions de personnages avec BERT (ANTOINE)
Usage de perceptron mono-couche pour prédire si l'embeddings de BERT correspond à une mention de personnage. S'il y a le temps : faire un perceptron multicouche pour faire la même tâche.

In [1]:
from datasets import load_dataset
from zmq.sugar import device

dataset = load_dataset("lattice-nlp/litbank-fr-PER")["train"].to_list()

print(dataset[0].keys())

dict_keys(['file_name', 'language', 'text', 'stanza_tokens', 'mentions', 'sentence_spans'])


### Exploration Structure Dataset

In [2]:
print(f"{len(dataset):,} documents\n")

example_document = dataset[0]

for key, value in example_document.items():
    print(f"{key:<20} {type(value)}    {repr(value)[:100]}")

32 documents

file_name            <class 'str'>    '1882-1883_Maupassant-Guy-de_Mademoiselle-Fifi-La-buche'
language             <class 'str'>    'fr'
text                 <class 'str'>    "LA BUCHE\n\nLe salon était petit, tout enveloppé de tentures épaisses, et discrètement odorant. Dan
stanza_tokens        <class 'int'>    2249
mentions             <class 'list'>    [{'onset': 278, 'offset': 296, 'COREF': 'deux_personnes', 'type': 'PER'}, {'onset': 297, 'offset': 3
sentence_spans       <class 'list'>    [[0, 10], [10, 94], [94, 313], [313, 762], [762, 879], [879, 905], [905, 1137], [1137, 1217], [1217,


In [3]:
print(f"{len(example_document["mentions"]):,} mentions\n")

example_mention = example_document["mentions"][0]
for key, value in example_mention.items():
    print(f"{key:<20} {type(value)}    {repr(value)[:100]}")

235 mentions

onset                <class 'int'>    278
offset               <class 'int'>    296
COREF                <class 'str'>    'deux_personnes'
type                 <class 'str'>    'PER'


### Ajouter le champ "text" pour chaque mention. Récupérer l'empans correspondant à `onset`:`offset`.

In [4]:
for document in dataset:
    for mention in document["mentions"]:
        mention["text"] = document["text"][mention["onset"]:mention["offset"]]

In [5]:
dataset[0]["mentions"][:10]

[{'onset': 278,
  'offset': 296,
  'COREF': 'deux_personnes',
  'type': 'PER',
  'text': 'les deux personnes'},
 {'onset': 297,
  'offset': 300,
  'COREF': 'deux_personnes',
  'type': 'PER',
  'text': 'qui'},
 {'onset': 313,
  'offset': 317,
  'COREF': 'vieille',
  'type': 'PER',
  'text': 'Elle'},
 {'onset': 320,
  'offset': 345,
  'COREF': 'vieille',
  'type': 'PER',
  'text': 'la maîtresse de la maison'},
 {'onset': 347,
  'offset': 375,
  'COREF': 'vieille',
  'type': 'PER',
  'text': 'une vieille à cheveux blancs'},
 {'onset': 382,
  'offset': 411,
  'COREF': 'vieille',
  'type': 'PER',
  'text': 'une de ces vieilles adorables'},
 {'onset': 684, 'offset': 688, 'COREF': 'M1', 'type': 'PER', 'text': 'vous'},
 {'onset': 762, 'offset': 765, 'COREF': 'paul', 'type': 'PER', 'text': 'Lui'},
 {'onset': 807,
  'offset': 836,
  'COREF': 'paul',
  'type': 'PER',
  'text': 'un ami de toutes les semaines'},
 {'onset': 838,
  'offset': 877,
  'COREF': 'paul',
  'type': 'PER',
  'text': "un comp

### Quel est le texte le plus long dans le dataset ?

In [6]:
document_len_dict = {document["file_name"]:len(document["text"]) for document in dataset}
document_len_dict = dict(
    sorted(document_len_dict.items(), key=lambda x: x[1], reverse=True)
)
longest_doc_file_name = list(document_len_dict.keys())[0]
print(longest_doc_file_name)

1832_Sand-George_Indiana


### Qui en sont les 5 personnages principaux

Voir : [https://fr.wikipedia.org/wiki/Indiana_(roman)#Personnages](https://fr.wikipedia.org/wiki/Indiana_(roman)#Personnages)

In [7]:
document = [document for document in dataset if document["file_name"] in document["file_name"]==longest_doc_file_name][0]

from collections import Counter

Counter([mention["COREF"] for mention in document["mentions"]]).most_common(5)

[('Indiana_Delmare_nee_de_Carvajal', 4929),
 ('Raymon_de_Ramiere', 3627),
 ('sir_Rodolphe_Brown_dit_Ralph', 2265),
 ('colonel_Delmare', 1292),
 ('Noun', 735)]

### Classification D'empans Textuels

#### BIO tags VS Classification d'empans candidats

BIO = difficultés pour mentions imbriquées

Classification d'empans = Trouver une façon de représenter un empans de plusieurs tokens (sous-mots)

In [8]:
from transformers import AutoTokenizer

model_name = "almanach/camembert-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

def extract_subword_dict_from_text(text, tokenizer):

    encoding = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    subword_ids = encoding["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(subword_ids)
    offsets = encoding["offset_mapping"]

    subwords_dict = [
        {   "ID": i,
            "subword_id": subword_id,
            "subword": token,
            "char_onset": start,
            "char_offset": end,
            "text": text[start:end]
        }
        for i, (subword_id, token, (start, end)) in enumerate(zip(subword_ids, tokens, offsets))
    ]
    return subwords_dict

text = "Par une soirée d'automne pluvieuse et fraîche, trois personnes rêveuses étaient gravement occupées, au fond d'un petit castel de la Brie, à regarder brûler les tisons du foyer et cheminer lentement l'aiguille de la pendule."
subwords_dict = extract_subword_dict_from_text(text, tokenizer)
# for subword in subwords_dict:
#     print(subword)

print([subword["subword"] for subword in subwords_dict])
print([subword["subword_id"] for subword in subwords_dict])

['▁Par', '▁une', '▁soirée', '▁d', "'", 'automne', '▁plu', 'vi', 'euse', '▁et', '▁fraîche', ',', '▁trois', '▁personnes', '▁rêve', 'uses', '▁étaient', '▁gravement', '▁occupée', 's', ',', '▁au', '▁fond', '▁d', "'", 'un', '▁petit', '▁cas', 'tel', '▁de', '▁la', '▁Bri', 'e', ',', '▁à', '▁regarder', '▁brûler', '▁les', '▁t', 'isons', '▁du', '▁foyer', '▁et', '▁chemin', 'er', '▁lentement', '▁l', "'", 'aiguille', '▁de', '▁la', '▁pendule', '.']
[372, 28, 1202, 18, 11, 5087, 5644, 1745, 1731, 14, 8623, 7, 303, 242, 1802, 2963, 530, 19066, 12450, 10, 7, 36, 729, 18, 11, 59, 226, 203, 2497, 8, 13, 4025, 35, 7, 15, 1483, 14580, 19, 271, 14481, 25, 5298, 14, 1111, 108, 6475, 17, 11, 21401, 8, 13, 26131, 9]


### Ajouter subwords à tous les documents du dataset

In [9]:
for document in dataset:
    document["subwords_dict"] = extract_subword_dict_from_text(document["text"], tokenizer)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2673 > 512). Running this sequence through the model will result in indexing errors


In [10]:
dataset[0]["subwords_dict"][:10]

[{'ID': 0,
  'subword_id': 1925,
  'subword': '▁LA',
  'char_onset': 0,
  'char_offset': 2,
  'text': 'LA'},
 {'ID': 1,
  'subword_id': 21299,
  'subword': '▁BU',
  'char_onset': 3,
  'char_offset': 5,
  'text': 'BU'},
 {'ID': 2,
  'subword_id': 5918,
  'subword': 'CHE',
  'char_onset': 5,
  'char_offset': 8,
  'text': 'CHE'},
 {'ID': 3,
  'subword_id': 54,
  'subword': '▁Le',
  'char_onset': 10,
  'char_offset': 12,
  'text': 'Le'},
 {'ID': 4,
  'subword_id': 1109,
  'subword': '▁salon',
  'char_onset': 13,
  'char_offset': 18,
  'text': 'salon'},
 {'ID': 5,
  'subword_id': 149,
  'subword': '▁était',
  'char_onset': 19,
  'char_offset': 24,
  'text': 'était'},
 {'ID': 6,
  'subword_id': 226,
  'subword': '▁petit',
  'char_onset': 25,
  'char_offset': 30,
  'text': 'petit'},
 {'ID': 7,
  'subword_id': 7,
  'subword': ',',
  'char_onset': 30,
  'char_offset': 31,
  'text': ','},
 {'ID': 8,
  'subword_id': 66,
  'subword': '▁tout',
  'char_onset': 32,
  'char_offset': 36,
  'text': 'tou

In [11]:
dataset[0]["mentions"]
mention = dataset[0]["mentions"][0]
mention

{'onset': 278,
 'offset': 296,
 'COREF': 'deux_personnes',
 'type': 'PER',
 'text': 'les deux personnes'}

In [12]:
from transformers import AutoModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())

model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

config = model.config

print("Model type:", config.model_type)

CUDA available: True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CamembertModel LOAD REPORT from: almanach/camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model type: camembert


In [13]:
max_context = tokenizer.model_max_length
print(f"Tokenizer limit: {max_context:,} subwords")

Tokenizer limit: 512 subwords


Max context = 8

Fenêtres sans chevauchement

```
Fenêtre 1: [A B C D E F G H]
Fenêtre 3:                 [I J K L M N O P]
```

Fenêtres avec chevauchement (ratio = 0.5)
```
Fenêtre 1: [A B C D E F G H]
Fenêtre 2:         [E F G H I J K L]
Fenêtre 3:                 [I J K L M N O P]
```

In [14]:
subwords_count = len(dataset[0]["subwords_dict"])
print(subwords_count)

overlap_ratio = 0.5
assert 0 <= overlap_ratio < 1
offset = int(max_context * (1 - overlap_ratio))

windows = []
start = 0

while start < subwords_count:
    end = min(start + max_context, subwords_count)
    windows.append((start, end))

    if end == subwords_count:
        break

    start += offset

print(f"{len(windows):,} windows")

2673
10 windows


In [15]:
subword_ids = [subword['subword_id'] for subword in dataset[0]["subwords_dict"]]

In [16]:
best_embeddings = {}
model.eval()

with torch.inference_mode():
    for start, end in windows:
        window_subword_ids = subword_ids[start:end]

        input_ids = torch.tensor(
            [window_subword_ids],
            dtype=torch.long,
            device=model.device
        )

        outputs = model(input_ids=input_ids)
        embeddings = outputs.last_hidden_state[0].cpu()

        window_center = (start + end - 1) / 2

        for local_idx, embedding in enumerate(embeddings):
            global_idx = start + local_idx
            distance = abs(global_idx - window_center)

            if (
                global_idx not in best_embeddings
                or distance < best_embeddings[global_idx]["distance"]
            ):
                best_embeddings[global_idx] = {
                    "embedding": embedding,
                    "distance": distance
                }

subword_embeddings = [
    best_embeddings[i]["embedding"]
    for i in range(len(subword_ids))
]

In [17]:
len(subword_embeddings), len(subword_embeddings[0])

(2673, 768)

In [22]:
import numpy as np
import pandas as pd
import umap

subwords = dataset[0]["subwords_dict"]
import numpy as np

X = np.stack([
    emb.numpy()
    for emb in subword_embeddings
]).astype(np.float32)

print("Embedding matrix:", X.shape)

assert len(X) == len(subwords), (
    f"Mismatch: {len(X)} embeddings vs {len(subwords)} subwords"
)

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=20,
    min_dist=0.5,
    metric="cosine",
    random_state=42
)

embedding_2d = reducer.fit_transform(X)

df = pd.DataFrame([
    {
        "ID": sw["ID"],
        "subword": sw["subword"],
        "text": sw["text"],
        "char_onset": sw["char_onset"],
        "char_offset": sw["char_offset"],
        "x": embedding_2d[i, 0],
        "y": embedding_2d[i, 1],
    }
    for i, sw in enumerate(subwords)
])

df.head()

Embedding matrix: (2673, 768)


/home/antoine/cours/CPES_2026-2027/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,ID,subword,text,char_onset,char_offset,x,y
0,0,▁LA,LA,0,2,2.913511,11.831820
1,1,▁BU,BU,3,5,15.214693,-1.110742
2,2,CHE,CHE,5,8,15.670230,-1.438724
3,3,▁Le,Le,10,12,15.485850,5.938319
4,4,▁salon,salon,13,18,13.212056,-0.076715


In [23]:
import plotly.express as px

fig = px.scatter(
    df,
    x="x",
    y="y",
    hover_data={
        "ID": True,
        "subword": True,
        "text": True,
        "char_onset": True,
        "char_offset": True,
        "x": False,
        "y": False,
    },
    title="UMAP projection of CamemBERT subword embeddings",
    labels={
        "x": "UMAP 1",
        "y": "UMAP 2",
    },
    template="plotly_white",
)

fig.update_traces(
    marker=dict(size=5, opacity=0.7)
)

fig.update_layout(
    width=1000,
    height=750,
)

fig.show()